In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

In [0]:
for field in df.schema.fields:
    if field.dataType == StringType:
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn('MAINTENANCE',
                   F.when(F.upper(col('MAINTENANCE')) == 'YES', F.lit(True))
                   .when(F.upper(col("MAINTENANCE")) == 'NO', F.lit(False))
                   .otherwise(None)
                   )

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_product_category')

In [0]:
%sql

SELECT COUNT(*)
FROM workspace.silver.erp_product_category
